In [1]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from tqdm import tqdm

In [2]:
ENTREPOT_PATH = "/home/tbadie/Bureau/data/data_entrepot_outils/"
PATH_TEMP = "/home/tbadie/Bureau/data/temp/typologie_dirodur/"
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

In [3]:
# Donnees d'intervention pour les dates de semis
tables = [
    'intervention_synthetise',
    'intervention_realise',
    'noeuds_realise',
    'noeuds_synthetise',
    'noeuds_synthetise_restructure',
    'connection_synthetise',
    'zone',
    'parcelle',
    'synthetise',
    'sdc'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 10/10 [00:32<00:00,  3.29s/it]


In [4]:
list_culture_choisies = [
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_000269f5-e64c-40f1-a581-1177bd937c0a',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_0002dbc4-5f42-445d-9944-b7a7f9c4a2e8',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_0003adb6-89d8-4695-9546-2166614548a4',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_000a5487-1845-4753-b983-ae8e089f3590',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_00450508-e65e-4577-bc54-83770e978308',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_004b130e-ea96-4c2b-8406-1403467dc0e3',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_fff7ec84-be83-4e51-8267-6882c22f129f',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_fff60f2f-f1d4-47f6-8d3e-d86310d73235',
    'fr.inra.agrosyst.api.entities.CroppingPlanEntry_ff26f543-5df1-4774-bb0f-ce263c2fe111'
]

In [5]:
noeuds_realise = donnees['noeuds_realise'].copy()
noeuds_realise = noeuds_realise.loc[noeuds_realise['culture_id'].isin(list_culture_choisies)]

noeuds_synthetise_restructure = donnees['noeuds_synthetise_restructure'].copy()
noeuds_synthetise_restructure = noeuds_synthetise_restructure.loc[noeuds_synthetise_restructure['culture_id'].isin(list_culture_choisies)]

noeuds_synthetise = donnees['noeuds_synthetise'].copy()
noeuds_synthetise = noeuds_synthetise.loc[noeuds_synthetise['id'].isin(noeuds_synthetise_restructure['id'])]

connection_synthetise = donnees['connection_synthetise'].copy()
connection_synthetise = connection_synthetise.loc[connection_synthetise['cible_noeuds_synthetise_id'].isin(noeuds_synthetise_restructure['id'])]

intervention_synthetise = donnees['intervention_synthetise'].copy()
intervention_synthetise = intervention_synthetise.loc[intervention_synthetise['connection_synthetise_id'].isin(connection_synthetise['id'])]

intervention_realise = donnees['intervention_realise'].copy()
intervention_realise = intervention_realise.loc[intervention_realise['noeuds_realise_id'].isin(noeuds_realise['id'])]

zone = donnees['zone'].copy()
zone = zone.loc[zone['id'].isin(noeuds_realise['zone_id'])]

parcelle = donnees['parcelle'].copy()
parcelle = parcelle.loc[parcelle['id'].isin(zone['parcelle_id'])]

synthetise = donnees['synthetise'].copy()
synthetise = synthetise.loc[synthetise['id'].isin(noeuds_synthetise['synthetise_id'])]

sdc = donnees['sdc'].copy()
sdc = sdc.loc[sdc['id'].isin(synthetise['sdc_id']) | (sdc['id'].isin(parcelle['sdc_id']))]

In [6]:
PATH_TEST='/home/tbadie/Bureau/catalogue_script_agrosyst/02_outils/tests/data/test_get_saison_semis/'

intervention_synthetise.to_csv(PATH_TEST+'intervention_synthetise.csv', index=False)
intervention_realise.to_csv(PATH_TEST+'intervention_realise.csv', index=False)
noeuds_realise.to_csv(PATH_TEST+'noeuds_realise.csv', index=False)
noeuds_synthetise.to_csv(PATH_TEST+'noeuds_synthetise.csv', index=False)
noeuds_synthetise_restructure.to_csv(PATH_TEST+'noeuds_synthetise_restructure.csv', index=False)
connection_synthetise.to_csv(PATH_TEST+'connection_synthetise.csv', index=False)
zone.to_csv(PATH_TEST+'zone.csv', index=False)
parcelle.to_csv(PATH_TEST+'parcelle.csv', index=False)
synthetise.to_csv(PATH_TEST+'synthetise.csv', index=False)
sdc.to_csv(PATH_TEST+'sdc.csv', index=False)

In [ ]:
del donnees
donnees = {}
import_dfs(tables, PATH_TEST, sep = ',', verbose=False)

100%|██████████| 10/10 [00:00<00:00, 362.55it/s]


In [10]:

def get_sowing_date(donnees) :
    """
    Calcule une date moyenne de semis par culture à partir des interventions date_din et date_début des interventions de semis en réalisées et synthétisées.
    """

    # Chargement des tables utiles
    intv_S = donnees['intervention_synthetise'][['id','type','date_debut','date_fin','concerne_ci','connection_synthetise_id']].copy()
    intv_R = donnees['intervention_realise'][['id','type','date_debut','date_fin','concerne_ci','noeuds_realise_id']].copy()
    conx_S = donnees['connection_synthetise'][['id','cible_noeuds_synthetise_id']].rename(columns={'id':'connection_synthetise_id', 'cible_noeuds_synthetise_id':'noeuds_synthetise_id'}).copy()
    noeud_w_culture_id_S = donnees['noeuds_synthetise_restructure'][['id','culture_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
    noeuds_w_culture_id_R = donnees['noeuds_realise'][['id','culture_id','zone_id']].rename(columns={'id':'noeuds_realise_id'}).copy()

    node_S = donnees['noeuds_synthetise'][['id','synthetise_id']].rename(columns={'id':'noeuds_synthetise_id'}).copy()
    synthe = donnees['synthetise'][['id','sdc_id']].rename(columns={'id':'synthetise_id'}).copy()
    zone = donnees['zone'][['id','parcelle_id']].rename(columns={'id':'zone_id'}).copy()
    parcelle = donnees['parcelle'][['id','sdc_id']].rename(columns={'id':'parcelle_id'}).copy()
    sdc = donnees['sdc'][['id','filiere']].rename(columns={'id':'sdc_id'}).copy()

    # Filtre sur les interventions de semis hors CI
    intv_S = intv_S.loc[(intv_S['type'] == 'SEMIS') & 
                        (intv_S['connection_synthetise_id'].notna()) & 
                        (intv_S['concerne_ci'] == 'f')]
    intv_R = intv_R.loc[(intv_R['type'] == 'SEMIS') & 
                        (intv_R['noeuds_realise_id'].notna()) & 
                        (intv_R['concerne_ci'] == 'f')]

    # Rattachement des cultures
    intv_S = intv_S.merge(conx_S, on='connection_synthetise_id', how='left')
    intv_S = intv_S.merge(noeud_w_culture_id_S, on='noeuds_synthetise_id', how='left')
    intv_R = intv_R.merge(noeuds_w_culture_id_R, on='noeuds_realise_id', how='left')


    def moyenne_dates(start_series, end_series, methode = 'R'):
        """ 
        Calcule la date moyenne entre date_debut et date_fin.
        """

        def parse_S_date(s):
            """ 
            Créer une date au format date. 
            Affecter une année pour avoir une vrai date. 
            On corrige le jour dans le cas des années bissextile et des erreurs de saisie 
            """
            j, m = map(int, s.split('/'))
            try :
                return datetime(2025, m, j)
            except ValueError:
                try : 
                    return datetime(2025, m, j-1)
                except ValueError:
                    return datetime(2025, m, j-2)
                
        def parse_R_date(s):
            """ 
            Créer une date au format date. 
            """
            y, m, j = map(int, s.split('-'))
            return datetime(y, m, j)
        
        # Cas ou les dates de fin sont antérieures aux dates de début
        if methode == 'S':
            start_dates = start_series.apply(parse_S_date)
            end_dates = end_series.apply(parse_S_date)

            mask = end_dates < start_dates
            end_dates.loc[mask] = end_dates.loc[mask].apply(lambda x: datetime(2026, x.month, x.day))

        elif methode == 'R':
            start_dates = start_series.apply(parse_R_date)
            end_dates = end_series.apply(parse_R_date)

        # Moyenne
        moyennes = start_dates + ((end_dates - start_dates) / 2)

        # Retourne la date au format jj/mm
        return moyennes.dt.strftime('%d/%m')

    # Date de semis moyenne par intervention
    intv_R['date_semis'] = moyenne_dates(intv_R['date_debut'], intv_R['date_fin'], methode='R')
    intv_S['date_semis'] = moyenne_dates(intv_S['date_debut'], intv_S['date_fin'], methode='S')

    # Fusion des données réalisées et synthétisées
    intv = pd.concat([intv_S, intv_R], ignore_index=True)

    # Rattachement au SDC puis filtrage sur GCPE
    intv = intv.merge(node_S, on='noeuds_synthetise_id',how='left')
    intv = intv.merge(synthe, on='synthetise_id',how='left')
    intv = intv.merge(zone, on='zone_id',how='left')
    intv = intv.merge(parcelle, on='parcelle_id',how='left')
    intv['sdc_id'] = np.where(intv['sdc_id_x'].notna(), intv['sdc_id_x'], intv['sdc_id_y'])
    intv = intv[['id','culture_id','date_semis','sdc_id']]
    intv = intv.merge(sdc, on='sdc_id',how='left')
    intv = intv.loc[intv['filiere'].isin(['POLYCULTURE_ELEVAGE','GRANDES_CULTURES'])]

    # Conversion en jour de l'année pour calcul circulaire
    # PS : Ajouter une année bissextile (2020) pour parser les dates
    intv['_dayofyear'] = pd.to_datetime(
        intv['date_semis'].astype(str) + '/2020',
        format='%d/%m/%Y'
    ).dt.dayofyear

    # Fonction pour trier un groupe en boucle circulaire
    def circular_mean_date(group):
        """
        Moyenne de dates en tenant compte du passage au nouvel an (ex: 30/12, 01/01, 02/01 => date moyenne = 01/01).
        """

        days = group['_dayofyear'].values
        if len(list(set(days))) == 1:
            return pd.Series({
                'dates_nbjr_triees': days.tolist(),
                'date_moyenne': group['date_semis'].values[0]
            })

        # Trouve l'écart maximal entre 2 dates consécutives (en boucle)
        sorted_days = np.sort(days)
        diffs = np.diff(np.r_[sorted_days, sorted_days[0] + 366])  # +366 pour gérer 29/02
        cut = np.argmax(diffs)+1
        # Réordonne en partant après la coupure
        order = np.roll(sorted_days, -cut)

        # Retourne la date moyenne
        asc_list = [y + order[0] if y < order[0] else y for y in order]
        moy = np.mean(asc_list)
        if moy > 366:
            moy -= 366
        date_moy = datetime(2020, 1, 1) + timedelta(days=moy-1)
        date_moy = date_moy.strftime('%d/%m') 

        return pd.Series({
            'dates_nbjr_triees': order.tolist(),
            'date_moyenne': date_moy
        })
    
    # Applique la fonction circular_mean_date et retourner l'objet
    return intv.groupby('culture_id', group_keys = False).apply(circular_mean_date, include_groups=False)

semis = get_sowing_date(donnees)

In [11]:
def transforme_date_en_saison(date_str):
    """ 
    A partir d'une date au format jj/mm, retourne la saison de semis correspondante.
    """

    j, m = map(int, date_str.split('/'))
    date = datetime(2024, m, j)
    if datetime(2024, 1, 16) <= date <= datetime(2024, 4, 15):
        return 'printemps'
    elif datetime(2024, 4, 16) <= date <= datetime(2024, 6, 15):
        return 'ete'
    elif datetime(2024, 6, 16) <= date <= datetime(2024, 9, 15):
        return 'automne'
    else:
        return 'hiver'

semis['saison_semis_detect_via_intv'] = semis['date_moyenne'].apply(transforme_date_en_saison)
semis.reset_index(inplace=True)

In [13]:
semis.to_csv(PATH_TEST+'output_semis.csv', index=False)